In [1]:

import os
import sys
import time
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from Helper_Scripts.calculate_metrics import (
    summarize_all_metrics,
    WINDOW as LOCAL_WINDOW,
    CWC_ETA,
)

from Helper_Scripts.models_TLMR import LSTMQuantileReg


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATHS = [
    "../Datasets/50Hertz.csv",
    "../Datasets/Amprion.csv",
    "../Datasets/TenneTTSO.csv",
    "../Datasets/TransnetBW.csv",
]

OUTPUT_CSV = "./results_all_methods_all_datasets.csv"
TRAJECTORY_DIR = "./trajectories/LSTM_0.016/"
SAVE_TRAJECTORIES = True

SEQ_LEN = 48
EPOCHS = 200
LR = 0.001
SEED = 42
BATCH_SIZE = 32

ALPHA = 0.1
TINIT = 5000
SPLIT = 0.5
GAMMA = 0.016

# Original 5,000-point rolling historical window.
HISTORY_WINDOW = TINIT

# Both models receive the SAME amount of training data.
TRAIN_WINDOW = int(round(SPLIT * HISTORY_WINDOW))
CAL_WINDOW = HISTORY_WINDOW - TRAIN_WINDOW

UPDATE_FREQUENCY = 6

# Preserve original hourly downsampling.
DOWNSAMPLE_STRIDE = 4

METHODS_TO_RUN = ["ACI"]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# QUANTILE REGRESSION LOSS
# ============================================================

def quantile_loss(y_true, y_pred, q):
    diff = y_true - y_pred

    return torch.mean(
        torch.maximum(
            q * diff,
            (q - 1) * diff
        )
    )


# ============================================================
# SEQUENCE DATASET
# ============================================================

class SeqDataset(Dataset):

    def __init__(self, series, seq_len):

        self.X = []
        self.y = []

        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])

        self.X = np.array(
            self.X,
            dtype=np.float32
        )

        self.y = np.array(
            self.y,
            dtype=np.float32
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(
                self.X[idx],
                dtype=torch.float32
            ),
            torch.tensor(
                self.y[idx],
                dtype=torch.float32
            )
        )


# ============================================================
# DATA LOADING
# FIX 1: NO FUTURE-DATA LEAKAGE IN SCALER
# ============================================================

def load_dataset(filepath, seq_len=SEQ_LEN):

    df = pd.read_csv(filepath)

    # Original dataset extraction logic.
    y = df.iloc[:, 1:].values.flatten().astype(float)

    # Original downsampling, now explicitly configurable.
    y = y[::DOWNSAMPLE_STRIDE]

    # FIX:
    # The first prediction target is at raw index:
    # TINIT + SEQ_LEN.
    #
    # Therefore only observations preceding that target
    # are used to fit the scaler.

    scaler_fit_end = TINIT + seq_len

    if len(y) <= scaler_fit_end:
        raise ValueError(
            f"{filepath}: insufficient observations. "
            f"Got {len(y)}, need more than {scaler_fit_end}."
        )

    scaler = StandardScaler()

    scaler.fit(
        y[:scaler_fit_end].reshape(-1, 1)
    )

    y_scaled = scaler.transform(
        y.reshape(-1, 1)
    ).flatten()

    full_dataset = SeqDataset(
        y_scaled,
        seq_len
    )

    return full_dataset, scaler, y, y_scaled


# ============================================================
# UTILITY
# ============================================================

def pad(arr, length):

    arr = np.array(arr, dtype=np.float32)

    if len(arr) < length:
        arr = np.concatenate([
            arr,
            np.full(
                length - len(arr),
                np.nan,
                dtype=np.float32
            )
        ])

    return arr


def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):

    beta_t = 1.0 - (
        np.sum(scores_cal <= score_t)
        / (len(scores_cal) + 1)
    )

    return float(
        np.clip(beta_t, 0.0, 1.0)
    )


# ============================================================
# CREATE INDEPENDENT LSTM MODELS
# ============================================================

def _fresh_models():

    model_lower = LSTMQuantileReg().to(device)
    model_upper = LSTMQuantileReg().to(device)

    opt_lower = torch.optim.Adam(
        model_lower.parameters(),
        lr=LR
    )

    opt_upper = torch.optim.Adam(
        model_upper.parameters(),
        lr=LR
    )

    return (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    )


# ============================================================
# SHARED TRAINING POLICY, SEPARATE MODEL INSTANCES
#
# Each experiment calls this function using its OWN
# LSTM instances and optimizers.
#
# FIX 2: Both train on TRAIN_WINDOW observations.
# FIX 3: Both retrain every UPDATE_FREQUENCY steps.
# ============================================================

def _maybe_train(
    t,
    tinit,
    X_train,
    y_train,
    model_lower,
    model_upper,
    opt_lower,
    opt_upper,
    quantiles
):

    if t == tinit:
        current_epochs = EPOCHS

    elif (t - tinit) % UPDATE_FREQUENCY == 0:
        current_epochs = 1

    else:
        current_epochs = 0

    if current_epochs == 0:
        return 0

    X_train_tensor = torch.tensor(
        X_train.reshape(-1, SEQ_LEN),
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train.reshape(-1, 1),
        dtype=torch.float32
    )

    train_ds = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=min(
            BATCH_SIZE,
            len(train_ds)
        ),
        shuffle=True
    )

    # ----------------------------
    # Train lower quantile model
    # ----------------------------

    model_lower.train()

    for _ in range(current_epochs):

        for X_b, y_b in train_loader:

            X_b = X_b.to(device)
            y_b = y_b.to(device)

            loss = quantile_loss(
                y_b,
                model_lower(X_b),
                quantiles[0]
            )

            opt_lower.zero_grad()
            loss.backward()
            opt_lower.step()

    # ----------------------------
    # Train upper quantile model
    # ----------------------------

    model_upper.train()

    for _ in range(current_epochs):

        for X_b, y_b in train_loader:

            X_b = X_b.to(device)
            y_b = y_b.to(device)

            loss = quantile_loss(
                y_b,
                model_upper(X_b),
                quantiles[1]
            )

            opt_upper.zero_grad()
            loss.backward()
            opt_upper.step()

    return current_epochs


# ============================================================
# COMMON HISTORICAL TRAINING INDICES
#
# At prediction time t:
#
# Historical window: [t - 5000, t)
#
# Training: first 2500 observations
# Calibration: next 2500 observations
#
# QR uses the training indices only.
# ACI uses training + calibration indices.
# ============================================================

def get_window_indices(t):

    start_idx = t - HISTORY_WINDOW

    train_end = start_idx + TRAIN_WINDOW

    train_points = np.arange(
        start_idx,
        train_end
    )

    cal_points = np.arange(
        train_end,
        t
    )

    return train_points, cal_points




def run_normal_qr(
    full_dataset,
    alpha=ALPHA,
    tinit=TINIT,
    splitSize=SPLIT
):

    # Reset seed for this independent experiment.
    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    y_u_preds = np.zeros(
        n_pred,
        dtype=np.float32
    )

    y_l_preds = np.zeros(
        n_pred,
        dtype=np.float32
    )

    y_true_vals = np.zeros(
        n_pred,
        dtype=np.float32
    )

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # QR trains its own models.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, _ = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Independent QR training.
        _maybe_train(
            t,
            tinit,
            X_train,
            y_train,
            model_lower,
            model_upper,
            opt_lower,
            opt_upper,
            quantiles
        )

        # One-step-ahead forecast.
        X_t_tensor = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():

            q_lo_t = float(
                model_lower(
                    X_t_tensor
                ).cpu().numpy()[0, 0]
            )

            q_hi_t = float(
                model_upper(
                    X_t_tensor
                ).cpu().numpy()[0, 0]
            )

        idx = t - tinit

        y_l_preds[idx] = q_lo_t
        y_u_preds[idx] = q_hi_t
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[Normal_QR] t={t}, "
                f"Interval=[{q_lo_t:.3f}, {q_hi_t:.3f}]"
            )

    empty = np.full(
        n_pred,
        np.nan,
        dtype=np.float32
    )

    return (
        empty.copy(),
        empty.copy(),
        y_u_preds,
        y_l_preds,
        y_true_vals
    )



def run_aci(
    full_dataset,
    alpha=ALPHA,
    gamma=GAMMA,
    tinit=TINIT,
    splitSize=SPLIT
):

    # Reset seed for this independent experiment.
    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    alphaTrajectory = np.zeros(
        n_pred,
        dtype=np.float32
    )

    adaptErrSeq = np.zeros(
        n_pred,
        dtype=np.float32
    )

    y_u_preds = np.zeros(
        n_pred,
        dtype=np.float32
    )

    y_l_preds = np.zeros(
        n_pred,
        dtype=np.float32
    )

    y_true_vals = np.zeros(
        n_pred,
        dtype=np.float32
    )

    alphat = alpha

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # ACI trains its own independent models.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, cal_points = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        X_cal = full_dataset.X[
            cal_points
        ].astype(np.float32)

        y_cal = full_dataset.y[
            cal_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Independent ACI model training.
        _maybe_train(
            t,
            tinit,
            X_train,
            y_train,
            model_lower,
            model_upper,
            opt_lower,
            opt_upper,
            quantiles
        )

        # -----------------------------------
        # Calibration predictions
        # -----------------------------------

        X_cal_tensor = torch.tensor(
            X_cal.reshape(-1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():

            y_l_cal = (
                model_lower(X_cal_tensor)
                .cpu()
                .numpy()
                .reshape(-1)
            )

            y_u_cal = (
                model_upper(X_cal_tensor)
                .cpu()
                .numpy()
                .reshape(-1)
            )

        # Original CQR nonconformity scores.
        scores_cal = np.maximum(
            y_l_cal - y_cal,
            y_cal - y_u_cal
        )

        # -----------------------------------
        # Current prediction
        # -----------------------------------

        X_t_tensor = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        with torch.no_grad():

            q_lo_t = float(
                model_lower(
                    X_t_tensor
                ).cpu().numpy()[0, 0]
            )

            q_hi_t = float(
                model_upper(
                    X_t_tensor
                ).cpu().numpy()[0, 0]
            )

        # -----------------------------------
        # Original ACI conformal correction
        # -----------------------------------

        alphat = np.clip(
            alphat,
            1e-6,
            1.0 - 1e-6
        )

        level_adapt = np.clip(
            (1.0 - alphat)
            * (1.0 + 1.0 / len(cal_points)),
            0.0,
            1.0
        )

        confQuantAdapt = float(
            np.quantile(
                scores_cal,
                level_adapt
            )
        )

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt

        # -----------------------------------
        # FIX 4:
        # NO ARTIFICIAL INTERVAL CLIPPING.
        #
        # Previously, intervals were clipped
        # using 5 * training-data range.
        #
        # That clipping has been removed.
        # -----------------------------------

        # -----------------------------------
        # Observe coverage error
        # -----------------------------------

        err_t = float(
            not (
                y_l_adapt <= y_t <= y_u_adapt
            )
        )

        # -----------------------------------
        # Original ACI alpha update
        # -----------------------------------

        alphat = np.clip(
            alphat + gamma * (alpha - err_t),
            1e-6,
            1.0 - 1e-6
        )

        # -----------------------------------
        # Save results
        # -----------------------------------

        idx = t - tinit

        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t

        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[ACI] t={t}, "
                f"alphat={alphat:.4f}, "
                f"err_t={err_t:.0f}"
            )

    return (
        alphaTrajectory,
        adaptErrSeq,
        y_u_preds,
        y_l_preds,
        y_true_vals
    )



METHOD_FUNCS = {
    "Normal_QR": run_normal_qr,
    "ACI": run_aci,
}




def run_one_method(
    dataset_name,
    method_name,
    full_dataset,
    scaler,
    **kwargs
):

    print(
        f"\n{'=' * 70}\n"
        f"{dataset_name} :: {method_name}\n"
        f"{'=' * 70}"
    )

    run_fn = METHOD_FUNCS[method_name]

    start_time = time.time()

    (
        alphaTraj,
        errSeq,
        y_u_preds,
        y_l_preds,
        y_true_vals
    ) = run_fn(
        full_dataset,
        **kwargs
    )

    time_taken = time.time() - start_time

    # Inverse transformation.
    y_u_inv = scaler.inverse_transform(
        y_u_preds.reshape(-1, 1)
    ).flatten()

    y_l_inv = scaler.inverse_transform(
        y_l_preds.reshape(-1, 1)
    ).flatten()

    y_true_inv = scaler.inverse_transform(
        y_true_vals.reshape(-1, 1)
    ).flatten()

    # Original metric implementation unchanged.
    metrics = summarize_all_metrics(
        y_true_inv,
        y_l_inv,
        y_u_inv,
        alpha=ALPHA,
        w=LOCAL_WINDOW,
        eta=CWC_ETA,
        time_taken=time_taken,
    )

    print(
        f"[{dataset_name}/{method_name}] "
        + " | ".join(
            f"{k}={v:.4f}"
            for k, v in metrics.items()
            if isinstance(v, float) and k != "time_sec"
        )
    )

    print(
        f"[{dataset_name}/{method_name}] "
        f"time_sec={time_taken:.2f}"
    )

    # -----------------------------------
    # Save prediction trajectories
    # -----------------------------------

    if SAVE_TRAJECTORIES:

        os.makedirs(
            TRAJECTORY_DIR,
            exist_ok=True
        )

        max_len = max(
            len(alphaTraj),
            len(errSeq),
            len(y_u_inv),
            len(y_l_inv),
            len(y_true_inv)
        )

        traj_df = pd.DataFrame({

            "step": np.arange(max_len),

            "alphaTrajectory": pad(
                alphaTraj,
                max_len
            ),

            "errSeq": pad(
                errSeq,
                max_len
            ),

            "y_u_pred": pad(
                y_u_inv,
                max_len
            ),

            "y_l_pred": pad(
                y_l_inv,
                max_len
            ),

            "y_true": pad(
                y_true_inv,
                max_len
            ),

        })

        safe_ds = os.path.splitext(
            os.path.basename(dataset_name)
        )[0]

        output_path = os.path.join(
            TRAJECTORY_DIR,
            f"{safe_ds}__{method_name}.csv"
        )

        traj_df.to_csv(
            output_path,
            index=False
        )

    return metrics


# ============================================================
# MAIN EXPERIMENT
# ============================================================

def main(dataset_paths):

    if not dataset_paths:

        print(
            "No datasets provided. "
            "Edit DATASET_PATHS."
        )

        sys.exit(1)

    all_rows = []

    for dataset_path in dataset_paths:

        print(
            f"\n\n########## Loading dataset: "
            f"{dataset_path} ##########"
        )

        (
            full_dataset,
            scaler,
            y_raw,
            y_scaled
        ) = load_dataset(
            dataset_path,
            seq_len=SEQ_LEN
        )

        print(
            f"Series length: {len(y_raw)}, "
            f"usable pairs: {len(full_dataset)}"
        )

        if len(full_dataset) <= TINIT:

            print(
                f"WARNING: {dataset_path} has "
                f"insufficient observations."
            )

            continue


        for method_name in METHODS_TO_RUN:

            kwargs = dict(
                alpha=ALPHA,
                tinit=TINIT,
                splitSize=SPLIT
            )

            if method_name == "ACI":
                kwargs["gamma"] = GAMMA

            metrics = run_one_method(
                dataset_path,
                method_name,
                full_dataset,
                scaler,
                **kwargs
            )

            row = {
                "dataset": os.path.basename(dataset_path),
                "method": method_name
            }

            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(
                all_rows
            ).to_csv(
                OUTPUT_CSV,
                index=False
            )

    results_df = pd.DataFrame(all_rows)

    col_order = [
        "dataset",
        "method",
        "PICP",
        "MPIW",
        "MLPICP",
        "LPICE",
        "LPIA",
        "LMPIW",
        "LCWC",
        "STD_LE",
        "n_local_windows",
        "time_sec"
    ]

    col_order = [
        c for c in col_order
        if c in results_df.columns
    ]

    results_df = results_df[
        col_order
        + [
            c for c in results_df.columns
            if c not in col_order
        ]
    ]

    results_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print(
        f"\nAll results written to: {OUTPUT_CSV}"
    )

    if SAVE_TRAJECTORIES:
        print(
            f"Trajectories: {TRAJECTORY_DIR}/"
        )

    print(
        "\n",
        results_df.to_string(index=False)
    )

    return results_df


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":

    main(DATASET_PATHS)


Device: cuda


########## Loading dataset: ../Datasets/50Hertz.csv ##########
Series length: 9528, usable pairs: 9480

../Datasets/50Hertz.csv :: ACI
[ACI] t=5000, alphat=0.1016, err_t=0
[ACI] t=5200, alphat=0.1016, err_t=0
[ACI] t=5400, alphat=0.1496, err_t=0
[ACI] t=5600, alphat=0.1176, err_t=0
[ACI] t=5800, alphat=0.1336, err_t=0
[ACI] t=6000, alphat=0.1336, err_t=0
[ACI] t=6200, alphat=0.2296, err_t=0
[ACI] t=6400, alphat=0.2136, err_t=0
[ACI] t=6600, alphat=0.1016, err_t=0
[ACI] t=6800, alphat=0.1656, err_t=0
[ACI] t=7000, alphat=0.2616, err_t=0
[ACI] t=7200, alphat=0.2456, err_t=0
[ACI] t=7400, alphat=0.1176, err_t=1
[ACI] t=7600, alphat=0.0536, err_t=0
[ACI] t=7800, alphat=0.1176, err_t=0
[ACI] t=8000, alphat=0.1656, err_t=0
[ACI] t=8200, alphat=0.1336, err_t=0
[ACI] t=8400, alphat=0.2456, err_t=0
[ACI] t=8600, alphat=0.1336, err_t=1
[ACI] t=8800, alphat=0.0536, err_t=0
[ACI] t=9000, alphat=0.1504, err_t=1
[ACI] t=9200, alphat=0.1344, err_t=0
[ACI] t=9400, alphat=0.1504, err_t=0

In [2]:



DATASET_PATHS = [
    "../Datasets/Location1.csv",
    "../Datasets/Location2.csv",
]

TINIT = 5000
SPLIT = 0.5

HISTORY_WINDOW = TINIT
TRAIN_WINDOW = int(round(TINIT * SPLIT))
CAL_WINDOW = TINIT - TRAIN_WINDOW

UPDATE_FREQUENCY = 6

METHODS_TO_RUN = ["ACI", "Normal_QR"]


# ============================================================
# FIX 1: LEAKAGE-FREE DATA LOADING
# ============================================================

def load_dataset(filepath, seq_len=SEQ_LEN):

    df = pd.read_csv(filepath)

    # Keep original location dataset selection.
    df = df.iloc[:10000].copy()

    y = df["Power"].to_numpy(
        dtype=np.float64
    )

    # First prediction target occurs at:
    # raw index TINIT + seq_len.
    #
    # Fit scaler only on observations available
    # before that prediction.

    scaler_fit_end = TINIT + seq_len

    if len(y) <= scaler_fit_end:
        raise ValueError(
            f"{filepath}: insufficient observations. "
            f"Got {len(y)}; need more than {scaler_fit_end}."
        )

    scaler = StandardScaler()

    scaler.fit(
        y[:scaler_fit_end].reshape(-1, 1)
    )

    y_scaled = scaler.transform(
        y.reshape(-1, 1)
    ).flatten()

    full_dataset = SeqDataset(
        y_scaled,
        seq_len
    )

    return full_dataset, scaler, y, y_scaled


# ============================================================
# FIX 2 & 3: IDENTICAL TRAINING-WINDOW POLICY
#
# At each time t:
#
# Historical observations: [t - 5000, t)
#
# Train: first 2500 observations
# Calibrate: next 2500 observations
#
# QR: uses training data only.
# ACI: uses training and calibration data.
#
# Both retrain every 6 steps.
# ============================================================

def get_window_indices(t):

    start_idx = t - HISTORY_WINDOW
    train_end = start_idx + TRAIN_WINDOW

    train_points = np.arange(
        start_idx,
        train_end
    )

    cal_points = np.arange(
        train_end,
        t
    )

    return train_points, cal_points


def set_seed(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# METHOD 1: NORMAL QR
# ============================================================

def run_normal_qr(
    full_dataset,
    alpha=ALPHA,
    tinit=TINIT,
    splitSize=SPLIT
):

    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    alphaTraj = np.full(
        n_pred,
        np.nan,
        dtype=np.float32
    )

    errSeq = np.full(
        n_pred,
        np.nan,
        dtype=np.float32
    )

    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # Independent QR models.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, _ = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Uses existing training function.
        # Initial: 200 epochs.
        # Every 6 steps: 1 epoch.
        _maybe_train(
            t,
            tinit,
            X_train,
            y_train,
            model_lower,
            model_upper,
            opt_lower,
            opt_upper,
            quantiles
        )

        X_t = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():

            q_lo = float(
                model_lower(X_t).cpu().numpy()[0, 0]
            )

            q_hi = float(
                model_upper(X_t).cpu().numpy()[0, 0]
            )

        idx = t - tinit

        y_l_preds[idx] = q_lo
        y_u_preds[idx] = q_hi
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[Normal_QR] t={t}, "
                f"Interval=[{q_lo:.3f}, {q_hi:.3f}]"
            )

    return (
        alphaTraj,
        errSeq,
        y_u_preds,
        y_l_preds,
        y_true_vals
    )


# ============================================================
# METHOD 2: ACI
# ============================================================

def run_aci(
    full_dataset,
    alpha=ALPHA,
    gamma=GAMMA,
    tinit=TINIT,
    splitSize=SPLIT
):

    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    alphaTrajectory = np.zeros(
        n_pred,
        dtype=np.float32
    )

    adaptErrSeq = np.zeros(
        n_pred,
        dtype=np.float32
    )

    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # Independent ACI models.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, cal_points = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        X_cal = full_dataset.X[
            cal_points
        ].astype(np.float32)

        y_cal = full_dataset.y[
            cal_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Train independently from QR.
        _maybe_train(
            t,
            tinit,
            X_train,
            y_train,
            model_lower,
            model_upper,
            opt_lower,
            opt_upper,
            quantiles
        )

        # Calibration predictions.
        X_cal_tensor = torch.tensor(
            X_cal.reshape(-1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():

            y_l_cal = (
                model_lower(X_cal_tensor)
                .cpu().numpy().reshape(-1)
            )

            y_u_cal = (
                model_upper(X_cal_tensor)
                .cpu().numpy().reshape(-1)
            )

        # Original nonconformity score.
        scores_cal = np.maximum(
            y_l_cal - y_cal,
            y_cal - y_u_cal
        )

        # Current forecast.
        X_t_tensor = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        with torch.no_grad():

            q_lo_t = float(
                model_lower(X_t_tensor)
                .cpu().numpy()[0, 0]
            )

            q_hi_t = float(
                model_upper(X_t_tensor)
                .cpu().numpy()[0, 0]
            )

        # Original ACI calibration rule.
        alphat = np.clip(
            alphat,
            1e-6,
            1.0 - 1e-6
        )

        level_adapt = np.clip(
            (1.0 - alphat)
            * (1.0 + 1.0 / len(cal_points)),
            0.0,
            1.0
        )

        confQuantAdapt = float(
            np.quantile(
                scores_cal,
                level_adapt
            )
        )

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt

        # FIX 4:
        # No artificial safety-margin clipping.

        err_t = float(
            not (
                y_l_adapt <= y_t <= y_u_adapt
            )
        )

        # Original ACI adaptation equation.
        alphat = np.clip(
            alphat + gamma * (alpha - err_t),
            1e-6,
            1.0 - 1e-6
        )

        idx = t - tinit

        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t

        y_l_preds[idx] = y_l_adapt
        y_u_preds[idx] = y_u_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[ACI] t={t}, "
                f"alphat={alphat:.4f}, "
                f"err_t={err_t:.0f}"
            )

    return (
        alphaTrajectory,
        adaptErrSeq,
        y_u_preds,
        y_l_preds,
        y_true_vals
    )


# ============================================================
# REGISTER UPDATED METHODS
# ============================================================

METHOD_FUNCS = {
    "Normal_QR": run_normal_qr,
    "ACI": run_aci
}


# ============================================================
# UPDATED MAIN
#
# QR no longer uses splitSize=1.
# Both experiments use SPLIT=0.5.
# ============================================================

def main(dataset_paths):

    if not dataset_paths:
        raise ValueError("No datasets provided.")

    all_rows = []

    for dataset_path in dataset_paths:

        print(
            f"\n########## Loading: "
            f"{dataset_path} ##########"
        )

        (
            full_dataset,
            scaler,
            y_raw,
            y_scaled
        ) = load_dataset(
            dataset_path,
            seq_len=SEQ_LEN
        )

        print(
            f"Series length: {len(y_raw)}, "
            f"usable pairs: {len(full_dataset)}"
        )

        if len(full_dataset) <= TINIT:
            print(f"Skipping {dataset_path}")
            continue

        for method_name in METHODS_TO_RUN:

            kwargs = {
                "alpha": ALPHA,
                "tinit": TINIT,
                "splitSize": SPLIT,
            }

            if method_name == "ACI":
                kwargs["gamma"] = GAMMA

            metrics = run_one_method(
                dataset_path,
                method_name,
                full_dataset,
                scaler,
                **kwargs
            )

            row = {
                "dataset": os.path.basename(dataset_path),
                "method": method_name
            }

            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(
                OUTPUT_CSV,
                index=False
            )

    results_df = pd.DataFrame(all_rows)

    col_order = [
        "dataset", "method",
        "PICP", "MPIW", "MLPICP",
        "LPICE", "LPIA", "LMPIW",
        "LCWC", "STD_LE",
        "n_local_windows", "time_sec"
    ]

    col_order = [
        c for c in col_order
        if c in results_df.columns
    ]

    results_df = results_df[
        col_order
        + [
            c for c in results_df.columns
            if c not in col_order
        ]
    ]

    results_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print("\n", results_df.to_string(index=False))

    return results_df


# ============================================================
# RUN LOCATION EXPERIMENTS
# ============================================================

location_results = main(DATASET_PATHS)



########## Loading: ../Datasets/Location1.csv ##########
Series length: 10000, usable pairs: 9952

../Datasets/Location1.csv :: ACI
[ACI] t=5000, alphat=0.1016, err_t=0
[ACI] t=5200, alphat=0.2136, err_t=0
[ACI] t=5400, alphat=0.1176, err_t=0
[ACI] t=5600, alphat=0.1336, err_t=0
[ACI] t=5800, alphat=0.1016, err_t=0
[ACI] t=6000, alphat=0.1656, err_t=0
[ACI] t=6200, alphat=0.0856, err_t=0
[ACI] t=6400, alphat=0.1176, err_t=0
[ACI] t=6600, alphat=0.0696, err_t=0
[ACI] t=6800, alphat=0.1496, err_t=0
[ACI] t=7000, alphat=0.0856, err_t=0
[ACI] t=7200, alphat=0.1016, err_t=0
[ACI] t=7400, alphat=0.1496, err_t=0
[ACI] t=7600, alphat=0.0856, err_t=0
[ACI] t=7800, alphat=0.0696, err_t=0
[ACI] t=8000, alphat=0.1176, err_t=0
[ACI] t=8200, alphat=0.0856, err_t=0
[ACI] t=8400, alphat=0.0696, err_t=0
[ACI] t=8600, alphat=0.1176, err_t=0
[ACI] t=8800, alphat=0.0696, err_t=0
[ACI] t=9000, alphat=0.0536, err_t=0
[ACI] t=9200, alphat=0.1496, err_t=0
[ACI] t=9400, alphat=0.1016, err_t=0
[ACI] t=9600, al

In [3]:

# ============================================================
# SOLAR PLANT 1 & 2 - CORRECTED QR AND ACI EXPERIMENTS
# Run after your existing helper-function definitions.
# ============================================================

DATASET_PATHS = [
    "../Datasets/Plant_1_Generation_Data.csv",
    "../Datasets/Plant_2_Generation_Data.csv",
]

TINIT = 4000
SPLIT = 0.5

HISTORY_WINDOW = TINIT
TRAIN_WINDOW = int(round(SPLIT * HISTORY_WINDOW))
CAL_WINDOW = HISTORY_WINDOW - TRAIN_WINDOW

UPDATE_FREQUENCY = 6
METHODS_TO_RUN = ["ACI", "Normal_QR"]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# FIX 1: LEAKAGE-FREE SOLAR DATA LOADING
# ============================================================

def load_dataset(filepath, seq_len=SEQ_LEN):

    df = pd.read_csv(filepath)

    # Preserve original first 10,000 rows.
    df = df.iloc[:10000].copy()

    print("Columns:", df.columns.tolist())

    # Preserve original DAILY_YIELD filtering.
    y = df.loc[
        df["DAILY_YIELD"].notna()
        & (df["DAILY_YIELD"] != 0),
        "DAILY_YIELD"
    ].to_numpy(dtype=np.float64)

    # First prediction target is at raw-series
    # position TINIT + seq_len.
    # Fit scaler only on preceding observations.

    scaler_fit_end = TINIT + seq_len

    if len(y) <= scaler_fit_end:
        raise ValueError(
            f"{filepath}: insufficient observations "
            f"after DAILY_YIELD filtering. "
            f"Available={len(y)}, "
            f"required>{scaler_fit_end}"
        )

    scaler = StandardScaler()

    scaler.fit(
        y[:scaler_fit_end].reshape(-1, 1)
    )

    y_scaled = scaler.transform(
        y.reshape(-1, 1)
    ).flatten()

    full_dataset = SeqDataset(
        y_scaled,
        seq_len
    )

    return full_dataset, scaler, y, y_scaled


# ============================================================
# FIX 2 & 3: MATCHED TRAINING-WINDOW POLICY
#
# Historical window: [t - 4000, t)
# Training: first 2000 samples
# Calibration: next 2000 samples
#
# QR uses training only.
# ACI uses training + calibration.
# ============================================================

def get_window_indices(t):

    start_idx = t - HISTORY_WINDOW
    train_end = start_idx + TRAIN_WINDOW

    train_points = np.arange(
        start_idx,
        train_end
    )

    cal_points = np.arange(
        train_end,
        t
    )

    return train_points, cal_points


# ============================================================
# METHOD 1: NORMAL QUANTILE REGRESSION
# ============================================================

def run_normal_qr(
    full_dataset,
    alpha=ALPHA,
    tinit=TINIT,
    splitSize=SPLIT
):

    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # Fresh, independent QR LSTMs.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, _ = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Initial 200 epochs, then 1 every 6 steps.
        _maybe_train(
            t, tinit,
            X_train, y_train,
            model_lower, model_upper,
            opt_lower, opt_upper,
            quantiles
        )

        X_t = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():
            q_lo = float(
                model_lower(X_t).item()
            )
            q_hi = float(
                model_upper(X_t).item()
            )

        idx = t - tinit

        y_l_preds[idx] = q_lo
        y_u_preds[idx] = q_hi
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[Normal_QR] t={t}, "
                f"Interval=[{q_lo:.3f}, {q_hi:.3f}]"
            )

    empty = np.full(
        n_pred,
        np.nan,
        dtype=np.float32
    )

    return (
        empty.copy(),
        empty.copy(),
        y_u_preds,
        y_l_preds,
        y_true_vals
    )


# ============================================================
# METHOD 2: ADAPTIVE CONFORMAL INFERENCE
# ============================================================

def run_aci(
    full_dataset,
    alpha=ALPHA,
    gamma=GAMMA,
    tinit=TINIT,
    splitSize=SPLIT
):

    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    alphaTrajectory = np.zeros(
        n_pred, dtype=np.float32
    )

    adaptErrSeq = np.zeros(
        n_pred, dtype=np.float32
    )

    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # Fresh, independent ACI LSTMs.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, cal_points = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        X_cal = full_dataset.X[
            cal_points
        ].astype(np.float32)

        y_cal = full_dataset.y[
            cal_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Train ACI LSTMs independently.
        _maybe_train(
            t, tinit,
            X_train, y_train,
            model_lower, model_upper,
            opt_lower, opt_upper,
            quantiles
        )

        # ----------------------------------------
        # Calibration predictions
        # ----------------------------------------

        X_cal_tensor = torch.tensor(
            X_cal.reshape(-1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():

            y_l_cal = (
                model_lower(X_cal_tensor)
                .cpu().numpy().reshape(-1)
            )

            y_u_cal = (
                model_upper(X_cal_tensor)
                .cpu().numpy().reshape(-1)
            )

        # Original CQR conformity score.
        scores_cal = np.maximum(
            y_l_cal - y_cal,
            y_cal - y_u_cal
        )

        # ----------------------------------------
        # Current one-step-ahead prediction
        # ----------------------------------------

        X_t_tensor = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        with torch.no_grad():

            q_lo_t = float(
                model_lower(X_t_tensor).item()
            )

            q_hi_t = float(
                model_upper(X_t_tensor).item()
            )

        # ----------------------------------------
        # Original ACI conformal calibration
        # ----------------------------------------

        alphat = np.clip(
            alphat,
            1e-6,
            1.0 - 1e-6
        )

        level_adapt = np.clip(
            (1.0 - alphat)
            * (1.0 + 1.0 / len(cal_points)),
            0.0,
            1.0
        )

        confQuantAdapt = float(
            np.quantile(
                scores_cal,
                level_adapt
            )
        )

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt

        # ----------------------------------------
        # FIX 4: NO ARTIFICIAL INTERVAL CLIPPING
        # ----------------------------------------

        # ----------------------------------------
        # Coverage error
        # ----------------------------------------

        err_t = float(
            not (
                y_l_adapt <= y_t <= y_u_adapt
            )
        )

        # ----------------------------------------
        # Original ACI alpha update
        # ----------------------------------------

        alphat = np.clip(
            alphat + gamma * (alpha - err_t),
            1e-6,
            1.0 - 1e-6
        )

        idx = t - tinit

        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t

        y_l_preds[idx] = y_l_adapt
        y_u_preds[idx] = y_u_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[ACI] t={t}, "
                f"alphat={alphat:.4f}, "
                f"err_t={err_t:.0f}"
            )

    return (
        alphaTrajectory,
        adaptErrSeq,
        y_u_preds,
        y_l_preds,
        y_true_vals
    )


# ============================================================
# REGISTER METHODS
# ============================================================

METHOD_FUNCS = {
    "Normal_QR": run_normal_qr,
    "ACI": run_aci
}


# ============================================================
# RUN EXPERIMENTS
# Reuses existing run_one_method() and metric functions.
# ============================================================

def main_solar(dataset_paths):

    all_rows = []

    for dataset_path in dataset_paths:

        print(
            f"\n{'=' * 70}\n"
            f"Dataset: {dataset_path}\n"
            f"{'=' * 70}"
        )

        (
            full_dataset,
            scaler,
            y_raw,
            y_scaled
        ) = load_dataset(
            dataset_path,
            seq_len=SEQ_LEN
        )

        print("Series length:", len(y_raw))
        print("Usable sequences:", len(full_dataset))
        print("Training window:", TRAIN_WINDOW)
        print("Calibration window:", CAL_WINDOW)

        if len(full_dataset) <= TINIT:
            print("Skipping: insufficient observations")
            continue

        for method_name in METHODS_TO_RUN:

            kwargs = {
                "alpha": ALPHA,
                "tinit": TINIT,
                "splitSize": SPLIT
            }

            if method_name == "ACI":
                kwargs["gamma"] = GAMMA

            # Existing driver handles:
            # - inverse scaling
            # - metrics
            # - trajectory CSV saving

            metrics = run_one_method(
                dataset_path,
                method_name,
                full_dataset,
                scaler,
                **kwargs
            )

            row = {
                "dataset": os.path.basename(dataset_path),
                "method": method_name
            }

            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(
                OUTPUT_CSV,
                index=False
            )

    results_df = pd.DataFrame(all_rows)

    col_order = [
        "dataset", "method",
        "PICP", "MPIW", "MLPICP",
        "LPICE", "LPIA", "LMPIW",
        "LCWC", "STD_LE",
        "n_local_windows", "time_sec"
    ]

    col_order = [
        c for c in col_order
        if c in results_df.columns
    ]

    results_df = results_df[
        col_order
        + [
            c for c in results_df.columns
            if c not in col_order
        ]
    ]

    results_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print("\nFINAL SOLAR RESULTS")
    print(results_df.to_string(index=False))

    return results_df


# ============================================================
# EXECUTE
# ============================================================

solar_results = main_solar(DATASET_PATHS)


Device: cuda

Dataset: ../Datasets/Plant_1_Generation_Data.csv
Columns: ['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']
Series length: 7379
Usable sequences: 7331
Training window: 2000
Calibration window: 2000

../Datasets/Plant_1_Generation_Data.csv :: ACI
[ACI] t=4000, alphat=0.0856, err_t=1
[ACI] t=4200, alphat=0.0224, err_t=0
[ACI] t=4400, alphat=0.0384, err_t=0
[ACI] t=4600, alphat=0.2464, err_t=0
[ACI] t=4800, alphat=0.5184, err_t=0
[ACI] t=5000, alphat=0.5344, err_t=0
[ACI] t=5200, alphat=0.4224, err_t=0
[ACI] t=5400, alphat=0.3904, err_t=0
[ACI] t=5600, alphat=0.3904, err_t=1
[ACI] t=5800, alphat=0.1824, err_t=0
[ACI] t=6000, alphat=0.0384, err_t=0
[ACI] t=6200, alphat=0.1648, err_t=0
[ACI] t=6400, alphat=0.2608, err_t=0
[ACI] t=6600, alphat=0.0848, err_t=0
[ACI] t=6800, alphat=0.0688, err_t=0
[ACI] t=7000, alphat=0.0224, err_t=0
[ACI] t=7200, alphat=0.0272, err_t=1
[../Datasets/Plant_1_Generation_Data.csv/ACI] PICP=0.8967 | MPIW=86

In [4]:

# ============================================================
# HYDROPOWER DATASET
# CORRECTED NORMAL QR AND ACI EXPERIMENTS
#
# Run after the existing notebook helper definitions.
#
# Critical fixes:
# 1. Remove normalization leakage.
# 2. Match QR / ACI training-set sizes.
# 3. Use consistent rolling retraining windows.
# 4. Remove artificial ACI interval clipping.
#
# QR and ACI use independently trained LSTMs.
# ============================================================

DATASET_PATHS = [
    "../Datasets/hydropower_dataset.csv",
]

TINIT = 200
SPLIT = 0.5

HISTORY_WINDOW = TINIT
TRAIN_WINDOW = int(round(TINIT * SPLIT))
CAL_WINDOW = TINIT - TRAIN_WINDOW

UPDATE_FREQUENCY = 6
METHODS_TO_RUN = ["ACI", "Normal_QR"]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# FIX 1: LEAKAGE-FREE HYDROPOWER DATA LOADING
# ============================================================

def load_dataset(filepath, seq_len=SEQ_LEN):

    df = pd.read_csv(filepath)

    # Preserve the original hydropower target.
    df["total_power_output"] = (
        df["Actual Day Peak (MW)"]
        + df["Actual Eve Peak (MW)"]
    )

    # Preserve original non-null / nonzero filtering.
    y = df.loc[
        df["total_power_output"].notna()
        & (df["total_power_output"] != 0),
        "total_power_output"
    ].to_numpy(dtype=np.float64)

    # First prediction target occurs at raw index
    # TINIT + seq_len.
    #
    # Fit the scaler using observations available
    # before the first prediction.

    scaler_fit_end = TINIT + seq_len

    if len(y) <= scaler_fit_end:
        raise ValueError(
            f"{filepath}: insufficient observations "
            f"after filtering. "
            f"Available={len(y)}, "
            f"required>{scaler_fit_end}"
        )

    scaler = StandardScaler()

    scaler.fit(
        y[:scaler_fit_end].reshape(-1, 1)
    )

    y_scaled = scaler.transform(
        y.reshape(-1, 1)
    ).flatten()

    full_dataset = SeqDataset(
        y_scaled,
        seq_len
    )

    return full_dataset, scaler, y, y_scaled


# ============================================================
# FIX 2 & 3: IDENTICAL TRAINING POLICY
#
# At prediction step t:
#
# Historical window: [t - 200, t)
# Training samples: first 100
# Calibration samples: next 100
#
# Normal QR: uses training samples only.
# ACI: uses training + calibration samples.
# ============================================================

def get_window_indices(t):

    start_idx = t - HISTORY_WINDOW
    train_end = start_idx + TRAIN_WINDOW

    train_points = np.arange(
        start_idx,
        train_end
    )

    cal_points = np.arange(
        train_end,
        t
    )

    return train_points, cal_points


# ============================================================
# METHOD 1: NORMAL QUANTILE REGRESSION
# ============================================================

def run_normal_qr(
    full_dataset,
    alpha=ALPHA,
    tinit=TINIT,
    splitSize=SPLIT
):

    # Independent initialization for QR.
    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    y_u_preds = np.zeros(
        n_pred, dtype=np.float32
    )

    y_l_preds = np.zeros(
        n_pred, dtype=np.float32
    )

    y_true_vals = np.zeros(
        n_pred, dtype=np.float32
    )

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # Fresh independent LSTMs for QR.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, _ = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Existing training helper:
        # Initial 200 epochs.
        # Subsequently 1 epoch every 6 steps.
        _maybe_train(
            t,
            tinit,
            X_train,
            y_train,
            model_lower,
            model_upper,
            opt_lower,
            opt_upper,
            quantiles
        )

        X_t_tensor = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():

            q_lo_t = float(
                model_lower(X_t_tensor).item()
            )

            q_hi_t = float(
                model_upper(X_t_tensor).item()
            )

        idx = t - tinit

        y_l_preds[idx] = q_lo_t
        y_u_preds[idx] = q_hi_t
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[Normal_QR] t={t}, "
                f"Interval=[{q_lo_t:.3f}, {q_hi_t:.3f}]"
            )

    empty = np.full(
        n_pred,
        np.nan,
        dtype=np.float32
    )

    return (
        empty.copy(),
        empty.copy(),
        y_u_preds,
        y_l_preds,
        y_true_vals
    )


# ============================================================
# METHOD 2: ADAPTIVE CONFORMAL INFERENCE
# ============================================================

def run_aci(
    full_dataset,
    alpha=ALPHA,
    gamma=GAMMA,
    tinit=TINIT,
    splitSize=SPLIT
):

    # Independent initialization for ACI.
    set_seed(SEED)

    T = len(full_dataset)
    n_pred = T - tinit

    alphaTrajectory = np.zeros(
        n_pred, dtype=np.float32
    )

    adaptErrSeq = np.zeros(
        n_pred, dtype=np.float32
    )

    y_u_preds = np.zeros(
        n_pred, dtype=np.float32
    )

    y_l_preds = np.zeros(
        n_pred, dtype=np.float32
    )

    y_true_vals = np.zeros(
        n_pred, dtype=np.float32
    )

    alphat = alpha

    quantiles = [
        alpha / 2.0,
        1.0 - alpha / 2.0
    ]

    # Fresh independent LSTMs for ACI.
    (
        model_lower,
        model_upper,
        opt_lower,
        opt_upper
    ) = _fresh_models()

    for t in range(tinit, T):

        train_points, cal_points = get_window_indices(t)

        X_train = full_dataset.X[
            train_points
        ].astype(np.float32)

        y_train = full_dataset.y[
            train_points
        ].astype(np.float32)

        X_cal = full_dataset.X[
            cal_points
        ].astype(np.float32)

        y_cal = full_dataset.y[
            cal_points
        ].astype(np.float32)

        y_t = float(full_dataset.y[t])

        # Independently train ACI LSTMs.
        _maybe_train(
            t,
            tinit,
            X_train,
            y_train,
            model_lower,
            model_upper,
            opt_lower,
            opt_upper,
            quantiles
        )

        # ----------------------------------------
        # Calibration predictions
        # ----------------------------------------

        X_cal_tensor = torch.tensor(
            X_cal.reshape(-1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        model_lower.eval()
        model_upper.eval()

        with torch.no_grad():

            y_l_cal = (
                model_lower(X_cal_tensor)
                .cpu().numpy().reshape(-1)
            )

            y_u_cal = (
                model_upper(X_cal_tensor)
                .cpu().numpy().reshape(-1)
            )

        # Original conformity scores.
        scores_cal = np.maximum(
            y_l_cal - y_cal,
            y_cal - y_u_cal
        )

        # ----------------------------------------
        # Current prediction
        # ----------------------------------------

        X_t_tensor = torch.tensor(
            full_dataset.X[t].reshape(1, SEQ_LEN),
            dtype=torch.float32,
            device=device
        )

        with torch.no_grad():

            q_lo_t = float(
                model_lower(X_t_tensor).item()
            )

            q_hi_t = float(
                model_upper(X_t_tensor).item()
            )

        # ----------------------------------------
        # Original ACI conformal calibration
        # ----------------------------------------

        alphat = np.clip(
            alphat,
            1e-6,
            1.0 - 1e-6
        )

        level_adapt = np.clip(
            (1.0 - alphat)
            * (1.0 + 1.0 / len(cal_points)),
            0.0,
            1.0
        )

        confQuantAdapt = float(
            np.quantile(
                scores_cal,
                level_adapt
            )
        )

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt

        # ----------------------------------------
        # FIX 4: NO ARTIFICIAL INTERVAL CLIPPING
        # ----------------------------------------

        # ----------------------------------------
        # Coverage error
        # ----------------------------------------

        err_t = float(
            not (
                y_l_adapt <= y_t <= y_u_adapt
            )
        )

        # ----------------------------------------
        # Original ACI alpha update
        # ----------------------------------------

        alphat = np.clip(
            alphat + gamma * (alpha - err_t),
            1e-6,
            1.0 - 1e-6
        )

        idx = t - tinit

        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t

        y_l_preds[idx] = y_l_adapt
        y_u_preds[idx] = y_u_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(
                f"[ACI] t={t}, "
                f"alphat={alphat:.4f}, "
                f"err_t={err_t:.0f}"
            )

    return (
        alphaTrajectory,
        adaptErrSeq,
        y_u_preds,
        y_l_preds,
        y_true_vals
    )


# ============================================================
# REGISTER CORRECTED METHODS
# ============================================================

METHOD_FUNCS = {
    "Normal_QR": run_normal_qr,
    "ACI": run_aci
}


# ============================================================
# DRIVER
# Reuses existing run_one_method() for metrics and CSVs.
# ============================================================

def main_hydropower(dataset_paths):

    all_rows = []

    for dataset_path in dataset_paths:

        print(
            f"\n{'=' * 70}\n"
            f"Dataset: {dataset_path}\n"
            f"{'=' * 70}"
        )

        (
            full_dataset,
            scaler,
            y_raw,
            y_scaled
        ) = load_dataset(
            dataset_path,
            seq_len=SEQ_LEN
        )

        print("Series length:", len(y_raw))
        print("Usable sequences:", len(full_dataset))
        print("Training window:", TRAIN_WINDOW)
        print("Calibration window:", CAL_WINDOW)

        if len(full_dataset) <= TINIT:
            print("Skipping: insufficient observations")
            continue

        for method_name in METHODS_TO_RUN:

            kwargs = {
                "alpha": ALPHA,
                "tinit": TINIT,
                "splitSize": SPLIT
            }

            if method_name == "ACI":
                kwargs["gamma"] = GAMMA

            metrics = run_one_method(
                dataset_path,
                method_name,
                full_dataset,
                scaler,
                **kwargs
            )

            row = {
                "dataset": os.path.basename(dataset_path),
                "method": method_name
            }

            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(
                OUTPUT_CSV,
                index=False
            )

    results_df = pd.DataFrame(all_rows)

    col_order = [
        "dataset", "method",
        "PICP", "MPIW", "MLPICP",
        "LPICE", "LPIA", "LMPIW",
        "LCWC", "STD_LE",
        "n_local_windows", "time_sec"
    ]

    col_order = [
        c for c in col_order
        if c in results_df.columns
    ]

    results_df = results_df[
        col_order
        + [
            c for c in results_df.columns
            if c not in col_order
        ]
    ]

    results_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print("\nFINAL HYDROPOWER RESULTS")
    print(results_df.to_string(index=False))

    return results_df


# ============================================================
# EXECUTE
# ============================================================

hydropower_results = main_hydropower(DATASET_PATHS)


Device: cuda

Dataset: ../Datasets/hydropower_dataset.csv
Series length: 1753
Usable sequences: 1705
Training window: 100
Calibration window: 100

../Datasets/hydropower_dataset.csv :: ACI
[ACI] t=200, alphat=0.1016, err_t=0
[ACI] t=400, alphat=0.0856, err_t=0
[ACI] t=600, alphat=0.1496, err_t=1
[ACI] t=800, alphat=0.1816, err_t=0
[ACI] t=1000, alphat=0.1456, err_t=0
[ACI] t=1200, alphat=0.0336, err_t=0
[ACI] t=1400, alphat=0.0656, err_t=0
[ACI] t=1600, alphat=0.1296, err_t=1
[../Datasets/hydropower_dataset.csv/ACI] PICP=0.8970 | MPIW=184.4223 | MLPICP=0.8996 | LPICE=0.0449 | LPIA=0.6059 | LMPIW=183.7945 | LCWC=1254234.2367 | STD_LE=0.0802
[../Datasets/hydropower_dataset.csv/ACI] time_sec=12.87

../Datasets/hydropower_dataset.csv :: Normal_QR
[Normal_QR] t=200, Interval=[-0.256, 0.185]
[Normal_QR] t=400, Interval=[-0.278, 1.808]
[Normal_QR] t=600, Interval=[-0.152, 2.894]
[Normal_QR] t=800, Interval=[-1.162, 0.157]
[Normal_QR] t=1000, Interval=[-0.388, 2.210]
[Normal_QR] t=1200, Interv

In [6]:

"""Audit ACI failure-point local errors using existing online trajectories.

No ACI rerun or synthetic fallback.

Quantile-crossing observations are excluded, not counted as failures.
A local window is eligible only when all 24 intervals are valid.

EWMA at i uses all valid local errors up to i, including LE_i,
with weights proportional to M**(i-k), using original time indices.
"""

from pathlib import Path
import numpy as np
import pandas as pd

ALPHA = 0.10
WINDOW = 24
M_DECAY = 0.95


def audit_one(path: Path, alpha=ALPHA, window=WINDOW, decay=M_DECAY):

    df = pd.read_csv(path)

    required = {'y_true', 'y_l_pred', 'y_u_pred'}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f'{path}: missing columns {sorted(missing)}'
        )

    if not (0 < alpha < 1 and window >= 2 and 0 < decay <= 1):
        raise ValueError(
            'Require 0<alpha<1, window>=2, 0<decay<=1'
        )

    if len(df) < window:
        raise ValueError(
            f'{path}: only {len(df)} rows (< window={window})'
        )

    for col in required:
        df[col] = pd.to_numeric(df[col], errors='raise')

    if not np.isfinite(
        df[list(required)].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f'{path}: missing or nonfinite interval observations'
        )

    # --------------------------------------------------------
    # FIX 1: Identify quantile crossing.
    # Do NOT count crossed intervals as failures.
    # --------------------------------------------------------

    y = df['y_true'].to_numpy(dtype=float)
    low = df['y_l_pred'].to_numpy(dtype=float)
    high = df['y_u_pred'].to_numpy(dtype=float)

    crossed = low > high
    valid = ~crossed

    crossed_count = int(crossed.sum())

    if crossed_count:
        print(
            f'WARNING {path.name}: '
            f'{crossed_count} crossed intervals excluded.'
        )

    # --------------------------------------------------------
    # Validate chronological order
    # --------------------------------------------------------

    if 'step' in df.columns:
        steps = pd.to_numeric(
            df['step'], errors='raise'
        ).to_numpy()

        if not np.all(np.diff(steps) == 1):
            raise ValueError(
                f'{path}: step must be consecutive and sorted'
            )
    else:
        steps = np.arange(len(df))

    # --------------------------------------------------------
    # FIX 2: Compute coverage errors.
    # Use stored online ACI errors if available.
    # Ignore crossed intervals.
    # --------------------------------------------------------

    computed_error = (
        (y < low) | (y > high)
    ).astype(np.int8)

    error = computed_error.copy()

    if 'errSeq' in df.columns:

        stored = pd.to_numeric(
            df['errSeq'], errors='raise'
        ).to_numpy(dtype=float)

        if (
            not np.isfinite(stored).all()
            or not np.isin(stored, [0, 1]).all()
        ):
            raise ValueError(
                f'{path}: errSeq contains invalid values'
            )

        mismatch = int(
            np.count_nonzero(
                (stored != computed_error) & valid
            )
        )

        if mismatch:
            print(
                f'WARNING {path.name}: '
                f'{mismatch} valid rows disagree with '
                f'saved interval coverage. '
                f'Using recorded errSeq.'
            )

        error = stored.astype(np.int8)

    # Crossed intervals are not evaluated.
    error = error.astype(float)
    error[crossed] = np.nan

    # --------------------------------------------------------
    # FIX 3: Full valid trailing windows only.
    #
    # If any observation in the last 24 is crossed,
    # the entire local window is ineligible.
    # --------------------------------------------------------

    covered = 1.0 - error

    local_coverage = (
        pd.Series(covered)
        .rolling(window, min_periods=window)
        .mean()
        .to_numpy()
    )

    local_error = np.maximum(
        (1 - alpha) - local_coverage,
        0.0
    )

    # Invalid or incomplete windows remain NaN.

    # --------------------------------------------------------
    # FIX 4: Exponentially weighted LE baseline.
    #
    # Original time indices are preserved.
    # Only valid local errors enter the baseline.
    #
    # LEbar_i =
    #   sum(valid k <= i) M**(i-k) * LE_k
    #   ----------------------------------
    #   sum(valid k <= i) M**(i-k)
    #
    # Current LE_i is included if valid.
    # --------------------------------------------------------

    baseline = np.full(len(df), np.nan)

    numerator = 0.0
    denominator = 0.0

    for i in range(window - 1, len(df)):

        numerator *= decay
        denominator *= decay

        if np.isfinite(local_error[i]):

            numerator += local_error[i]
            denominator += 1.0

            baseline[i] = numerator / denominator

    # --------------------------------------------------------
    # FIX 5: Eligible failures only.
    #
    # Exclude crossed observations and invalid windows.
    # --------------------------------------------------------

    eligible = (
        (np.arange(len(df)) >= window - 1)
        & valid
        & np.isfinite(local_error)
        & np.isfinite(baseline)
    )

    failure = (error == 1) & eligible

    # STRICT >
    above = (
        local_error[failure] > baseline[failure]
    )

    gap = (
        local_error[failure] - baseline[failure]
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    result = {
        'Dataset': path.stem.replace('__ACI', ''),
        'N_total': int(len(df)),
        'N_crossed_intervals': crossed_count,
        'N_failures_all': int(
            np.nansum(error == 1)
        ),
        'N_failures_eligible': int(failure.sum()),
        'N_LE_above_LEbar': int(above.sum()),
        'Pct_above': (
            100 * float(above.mean())
            if failure.any() else np.nan
        ),
        'Mean_LE_failure': (
            float(local_error[failure].mean())
            if failure.any() else np.nan
        ),
        'Mean_LEbar_failure': (
            float(baseline[failure].mean())
            if failure.any() else np.nan
        ),
        'Delta': (
            float(gap.mean())
            if failure.any() else np.nan
        ),
    }

    details = pd.DataFrame({
        'step': steps[failure],
        'LE_L': local_error[failure],
        'LEbar_L': baseline[failure],
        'Delta': gap,
        'LE_gt_LEbar': above.astype(int),
    })

    return result, details


def main():

    # Original nine dataset paths unchanged.

    files = [
        Path('./trajectories/LSTM_0.016/50Hertz__ACI.csv'),

        Path('./trajectories/LSTM_0.016/Amprion__ACI.csv'),

        Path('./trajectories/LSTM_0.016/TenneTTSO__ACI.csv'),

        Path('./trajectories/LSTM_0.016/TransnetBW__ACI.csv'),

        Path('./trajectories/LSTM_0.016/hydropower_dataset__ACI.csv'),

        Path('./trajectories/LSTM_0.016/Location1__ACI.csv'),

        Path('./trajectories/LSTM_0.016/Location2__ACI.csv'),

        Path('./trajectories/LSTM_0.016/Plant_1_Generation_Data__ACI.csv'),

        Path('./trajectories/LSTM_0.016/Plant_2_Generation_Data__ACI.csv'),
    ]

    output_dir = Path('./aci_failure_audit')

    alpha = ALPHA
    window = WINDOW
    decay = M_DECAY

    for p in files:
        if not p.is_file():
            raise FileNotFoundError(p)

    output_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for p in files:

        print(f'\nProcessing: {p.name}')

        summary, details = audit_one(
            p, alpha, window, decay
        )

        rows.append(summary)

        details.to_csv(
            output_dir / f'{p.stem}_failure_points.csv',
            index=False
        )

    res = pd.DataFrame(rows)

    res.to_csv(
        output_dir / 'aci_failure_summary.csv',
        index=False
    )

    print(
        res.to_string(
            index=False,
            float_format=lambda x: f'{x:.6f}'
        )
    )

    if len(res) > 1:

        total_above = int(
            res['N_LE_above_LEbar'].sum()
        )

        total_failures = int(
            res['N_failures_eligible'].sum()
        )

        weighted_pct = (
            100 * total_above / total_failures
            if total_failures else np.nan
        )

        # Pooled mean difference across all failures.
        pooled_delta = (
            np.average(
                res.loc[
                    res.N_failures_eligible > 0,
                    'Delta'
                ],
                weights=res.loc[
                    res.N_failures_eligible > 0,
                    'N_failures_eligible'
                ]
            )
            if total_failures else np.nan
        )

        print(
            f'\nPooled count: {total_above}'
            f'/{total_failures}'
            f' ({weighted_pct:.2f}%).'
        )

        print(
            f'Pooled mean Delta: {pooled_delta:.6f}'
        )

    print(
        f'\nSaved CSV outputs to: '
        f'{output_dir.resolve()}'
    )


if __name__ == '__main__':
    main()



Processing: 50Hertz__ACI.csv
WARNING 50Hertz__ACI.csv: 2 crossed intervals excluded.

Processing: Amprion__ACI.csv
WARNING Amprion__ACI.csv: 2 crossed intervals excluded.

Processing: TenneTTSO__ACI.csv
WARNING TenneTTSO__ACI.csv: 3 crossed intervals excluded.

Processing: TransnetBW__ACI.csv

Processing: hydropower_dataset__ACI.csv

Processing: Location1__ACI.csv
WARNING Location1__ACI.csv: 6 crossed intervals excluded.

Processing: Location2__ACI.csv
WARNING Location2__ACI.csv: 6 crossed intervals excluded.

Processing: Plant_1_Generation_Data__ACI.csv
WARNING Plant_1_Generation_Data__ACI.csv: 12 crossed intervals excluded.
WARNING Plant_1_Generation_Data__ACI.csv: 1 valid rows disagree with saved interval coverage. Using recorded errSeq.

Processing: Plant_2_Generation_Data__ACI.csv
WARNING Plant_2_Generation_Data__ACI.csv: 2 crossed intervals excluded.
                Dataset  N_total  N_crossed_intervals  N_failures_all  N_failures_eligible  N_LE_above_LEbar  Pct_above  Mean_LE_f